In [ ]:
import math
import numpy as np
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import ProgressBarCallback


# =========================
#  Custom Projectile Env
# =========================

class ProjectileEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self):
        super().__init__()

        # Physics
        self.g = 9.81
        self.d_min = 10.0
        self.d_max = 200.0

        # Angle & speed bounds
        self.theta_min = 0.1
        self.theta_max = math.pi / 2 - 0.1
        self.v_min = 5.0
        self.v_max = 50.0

        # Reward shaping
        self.lambda_v = 0.05
        self.tolerance = 0.1
        self.hit_bonus = 100.0

        # Observation: normalized target distance
        self.observation_space = spaces.Box(
            low=np.array([0.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
        )

        # Action: [angle_action, power_action] in [-1, 1]^2
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(2,), dtype=np.float32
        )

        self.target = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.target = np.random.uniform(self.d_min, self.d_max)
        obs = np.array([self.target / self.d_max], dtype=np.float32)
        return obs, {}

    def step(self, action):
        a_theta, a_v = np.clip(action, -1.0, 1.0)

        # Map action → physical parameters
        theta = self.theta_min + (a_theta + 1) / 2 * (self.theta_max - self.theta_min)
        v = self.v_min + (a_v + 1) / 2 * (self.v_max - self.v_min)

        # Projectile distance
        x_impact = (v**2 / self.g) * np.sin(2 * theta)
        error = abs(x_impact - self.target)

        # Reward: accuracy + light penalty on high power + hit bonus
        reward = -error - self.lambda_v * (v / self.v_max)
        if error < self.tolerance:
            reward += self.hit_bonus

        terminated = True  # single-step episode
        truncated = False
        obs = np.array([0.0], dtype=np.float32)  # dummy, episode ends

        info = {
            "target": self.target,
            "impact": x_impact,
            "theta": theta,
            "v": v,
            "error": error,
        }

        return obs, reward, terminated, truncated, info


# =========================
#  Training + Evaluation
# =========================

def main():
    env = ProjectileEnv()

    # SAC agent
    model = SAC(
        policy="MlpPolicy",
        env=env,
        verbose=1,
        learning_rate=3e-4,
        buffer_size=5000,
        batch_size=128,
        gamma=0.99,
        tau=0.005,
        train_freq=1,
        gradient_steps=1,
    )

    # Train
    model.learn(total_timesteps=100000, callback=ProgressBarCallback())
    model.save("sac_projectile")
    print("\nTraining complete. Saved as 'sac_projectile'.")

main()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


c:\Users\thap_as\AppData\Local\miniconda3\envs\thesis\Lib\site-packages\rich\live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1        |
|    ep_rew_mean     | -78.5    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 225      |
|    time_elapsed    | 0        |
|    total_timesteps | 4        |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1        |
|    ep_rew_mean     | -68.1    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 371      |
|    time_elapsed    | 0        |
|    total_timesteps | 8        |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1        |
|    ep_rew_mean     | -67.6    |
| time/              |          |
|    episodes        | 12       |
|    fps             | 496      |
|    time_elapsed    | 0        |
|    total_timesteps | 12       |
--------------

In [13]:
# Evaluate
print("\n=== Evaluation on fixed targets ===")
test_targets = [15,20,25,35,40,45, 30, 50, 80, 95]
env = ProjectileEnv()
model = SAC.load("sac_projectile", env=env)
for i in range(5):
    for tgt in test_targets:
        env.target = tgt
        obs = np.array([tgt / env.d_max], dtype=np.float32)
        action, _ = model.predict(obs, deterministic=True)
        _, _, _, _, info = env.step(action)

        print(
            f"Target: {tgt:5.1f} | Impact: {info['impact']:.2f} "
            f"| Error: {info['error']:.3f} | θ={info['theta']:.3f} rad | v={info['v']:.2f}"
            f"| Error Percent: {info['error']/tgt*100:.2f}%"
        )


=== Evaluation on fixed targets ===
Target:  15.0 | Impact: 16.30 | Error: 1.296 | θ=0.816 rad | v=12.66| Error Percent: 8.64%
Target:  20.0 | Impact: 22.13 | Error: 2.127 | θ=0.807 rad | v=14.74| Error Percent: 10.64%
Target:  25.0 | Impact: 27.26 | Error: 2.264 | θ=0.793 rad | v=16.35| Error Percent: 9.05%
Target:  35.0 | Impact: 37.68 | Error: 2.681 | θ=0.783 rad | v=19.23| Error Percent: 7.66%
Target:  40.0 | Impact: 42.74 | Error: 2.742 | θ=0.786 rad | v=20.48| Error Percent: 6.86%
Target:  45.0 | Impact: 47.81 | Error: 2.806 | θ=0.789 rad | v=21.66| Error Percent: 6.24%
Target:  30.0 | Impact: 33.13 | Error: 3.133 | θ=0.779 rad | v=18.03| Error Percent: 10.44%
Target:  50.0 | Impact: 52.69 | Error: 2.685 | θ=0.792 rad | v=22.74| Error Percent: 5.37%
Target:  80.0 | Impact: 83.35 | Error: 3.348 | θ=0.802 rad | v=28.60| Error Percent: 4.19%
Target:  95.0 | Impact: 98.60 | Error: 3.603 | θ=0.797 rad | v=31.11| Error Percent: 3.79%
Target:  15.0 | Impact: 16.30 | Error: 1.296 | θ=0.